# SUMM-Lens · 长文档摘要的零训练推理期增强

**主干**：Qwen2.5-1.5B-Instruct（2024，Apache 2.0，HF 直接加载）  
**模块**：CoD（Chain-of-Density 提示）+ NLR（NLI 重排）  
**数据集**：arXiv / PubMed

本 notebook 兼容 **本地 VSCode** 与 **Colab 免费版（T4）**：

- **Colab**：第 1 个 cell 会自动检测环境并 `git clone` 仓库 → `cd` 进去 → `pip install`。**首次运行前请把下面 `REPO_URL` 改成你的 GitHub 仓库地址**（私库则在 URL 中嵌入 token：`https://<TOKEN>@github.com/owner/repo.git`）。
- **本地 VSCode**：第 1 个 cell 检测到不在 Colab，会自动跳过 clone 与安装，直接用当前目录。

运行顺序：
1. 环境引导（Colab 自动 clone / 本地跳过）
2. 数据集预览
3. 单模型冒烟测试
4. CoD / NLR 模块演示
5. 跑 4 配置消融（Vanilla / +CoD / +NLR / +Both）
6. 可视化

## 1. 环境检查

In [ ]:
# ── 环境引导：Colab 自动 clone + install，本地直接跳过 ──────────────────
#
# 修改为你自己的仓库地址；私库可写成
#   https://<GITHUB_TOKEN>@github.com/your-name/SUMM-Lens.git
REPO_URL  = "https://github.com/zryshuaige/SUMM-Lens.git"
REPO_DIR  = "SUMM-Lens"   # clone 后的目录名
BRANCH    = "main"

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1) clone 仓库到 /content/
    if not os.path.isdir(f"/content/{REPO_DIR}"):
        print(f"[Colab] cloning {REPO_URL} ...")
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
            cwd="/content",
        )
    # 2) cd 进仓库根目录（之后所有相对路径都以此为基准）
    os.chdir(f"/content/{REPO_DIR}")
    print(f"[Colab] cwd = {os.getcwd()}")

    # 3) 安装依赖（Colab 已自带 torch，所以只装其余）
    print("[Colab] installing requirements ...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    )

    # 4) 中文字体（matplotlib 渲染中文图表需要）
    print("[Colab] installing CJK fonts ...")
    subprocess.check_call(
        ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"],
        stdout=subprocess.DEVNULL,
    )
    # 让 matplotlib 重新扫一遍系统字体
    import matplotlib.font_manager as fm
    fm._load_fontmanager(try_read_cache=False) if hasattr(fm, "_load_fontmanager") else fm.findSystemFonts()
else:
    # 本地：notebook 一般在 <repo>/notebooks/run.ipynb，cd 到仓库根
    nb_dir = os.path.abspath(os.getcwd())
    if os.path.basename(nb_dir) == "notebooks":
        os.chdir(os.path.dirname(nb_dir))
    print(f"[Local] cwd = {os.getcwd()}")

# 公共：把 src/ 加入 import 路径
SRC = os.path.join(os.getcwd(), "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# 强制 HuggingFace 镜像（中国大陆友好；不在国内可注释掉）
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

In [ ]:
import torch, transformers, datasets

print(f"Python   : {sys.version.split()[0]}")
print(f"PyTorch  : {torch.__version__}")
print(f"Transfor : {transformers.__version__}")
print(f"Datasets : {datasets.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}  "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")

## 2. 数据集预览

In [ ]:
from data_utils import load_arxiv_dataset, INPUT_FIELD, TARGET_FIELD

ds = load_arxiv_dataset()
print({k: len(v) for k, v in ds.items()})

sample = ds['test'][0]
print('\n--- ARTICLE (first 600 chars) ---')
print(sample[INPUT_FIELD][:600], '...')
print('\n--- REFERENCE ABSTRACT ---')
print(sample[TARGET_FIELD])

## 3. 单模型冒烟测试

先验证 BART 与 Qwen2.5 都能在你的环境上跑通。每个模型只跑 3 个样本。

In [ ]:
from evaluate import evaluate_model

# BART 走 seq2seq 路径
bart_res, bart_summaries, refs = evaluate_model(
    model_name='bart-large-cnn',
    dataset_name='arxiv',
    num_test_samples=3,
    output_dir='results/notebook',
    compute_bertscore=False,
    compute_meteor=False,
)
print('BART R1 =', bart_res['rouge']['rouge1']['fmeasure'])

In [ ]:
# Qwen2.5 走 causal-LM 路径，零样本提示
qwen_res, qwen_summaries, _ = evaluate_model(
    model_name='qwen2.5-1.5b',
    dataset_name='arxiv',
    num_test_samples=3,
    output_dir='results/notebook',
    compute_bertscore=False,
    compute_meteor=False,
)
print('Qwen2.5 R1 =', qwen_res['rouge']['rouge1']['fmeasure'])

## 4. CoD 与 NLR 模块单步演示

In [ ]:
from methods.llm_summarizer import CausalLMSummarizer
from methods.cod import chain_of_density
from methods.nli_rerank import NLIReranker, nli_rerank
from methods.prompts import get_prompt

summarizer = CausalLMSummarizer('Qwen/Qwen2.5-1.5B-Instruct', max_new_tokens=320)
reranker = NLIReranker()  # 复用 hallucination.HallucinationDetector

article = ds['test'][0][INPUT_FIELD]
reference = ds['test'][0][TARGET_FIELD]

In [ ]:
vanilla = summarizer.summarize(get_prompt('arxiv', article), n=1, temperature=0.0)[0]
cod_only = chain_of_density(summarizer, article, dataset='arxiv', num_iters=3)
nlr_only, nlr_info = nli_rerank(summarizer, article, dataset='arxiv', n_candidates=4, reranker=reranker, use_cod=False)
both, both_info = nli_rerank(summarizer, article, dataset='arxiv', n_candidates=4, reranker=reranker, use_cod=True, cod_iters=3)

print('Vanilla candidates NLI scores:', [f'{s:.3f}' for s in nlr_info['scores']])
print('CoD-densified  NLI scores   :', [f'{s:.3f}' for s in both_info['scores']])

In [ ]:
from IPython.display import HTML
from visualization import render_side_by_side

HTML(render_side_by_side(
    article, reference,
    {'Vanilla': vanilla, '+CoD': cod_only, '+NLR': nlr_only, '+CoD+NLR': both},
    max_article_chars=2000,
))

## 5. 跑完整消融（4 配置）

下面的 cell 大约会用 **5–15 分钟**（CPU）/ **1–3 分钟**（T4 GPU），处理 20 条样本。生产实验请把 `num_test` 调到 100。

In [ ]:
from run_experiments import run_ablation

ablation_results = run_ablation(
    dataset='arxiv',
    num_test=20,
    output_dir='results/notebook_ablation',
    compute_bertscore=False,
    compute_meteor=False,
)
for k, r in ablation_results.items():
    if 'error' in r:
        print(f'  {k}: ERROR {r["error"][:80]}')
    else:
        rouge = r['rouge']
        print(f'  {k:18s}  R1={rouge["rouge1"]["fmeasure"]:.4f}  R2={rouge["rouge2"]["fmeasure"]:.4f}  RL={rouge["rougeL"]["fmeasure"]:.4f}')

## 6. 可视化

In [ ]:
from analyze import plot_ablation_comparison

df = plot_ablation_comparison(ablation_results, 'results/notebook_ablation/figures')
df

In [ ]:
from IPython.display import Image
Image('results/notebook_ablation/figures/ablation_comparison.png')

## 7. 下一步

想跑完整 baseline 阶梯（5 个 baseline 模型 + 4 个消融配置）+ 出全部图：

```bash
python src/run_experiments.py --mode all --dataset arxiv --num_test 100
python src/run_experiments.py --mode all --dataset pubmed --num_test 100
```

结果会写入 `results/<timestamp>/`。